# Travel Analytics Capstone — Project Setup
**Project summary:** This notebook develops one component of a travel analytics system using the supplied CSV datasets. The broader project combines flight-price regression, gender classification, hotel recommendations, a Flask REST API, a Streamlit dashboard, MLflow tracking, Docker, Kubernetes, Airflow and CI/CD.

**GitHub repository:** `https://github.com/YOUR-USERNAME/travel-analytics-capstone`

> Before submission, replace the placeholder above with the one repository URL containing the full project. Keep all three notebooks pointed to the same repository.

In [ ]:
# In Google Colab, upload the CSVs in the Files panel or mount Google Drive.
# If the repository ZIP is uploaded to Colab, unzip it and set DATA_DIR accordingly.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("/content/data")  # Change to the folder containing flights.csv, hotels.csv, users.csv
if not (DATA_DIR / "flights.csv").exists():
    # Local notebook fallback: place CSVs in ./data
    DATA_DIR = Path("data")

flights = pd.read_csv(DATA_DIR / "flights.csv")
hotels = pd.read_csv(DATA_DIR / "hotels.csv")
users = pd.read_csv(DATA_DIR / "users.csv")

print("Flights:", flights.shape)
print("Hotels:", hotels.shape)
print("Users:", users.shape)
display(flights.head())
display(hotels.head())
display(users.head())

## Data understanding and quality checks

The flight dataset contains route, flight type, duration, distance, agency, date and observed price. The hotel dataset contains hotel, place, stay duration, daily price and total price. The users dataset contains age, company and a recorded gender label.

The first checks validate schema, duplicates, missing values, numeric types and date parsing. These checks matter because model quality depends on consistent input definitions; a pipeline that silently changes column types can fail after deployment.

In [ ]:
for name, frame in {"flights": flights, "hotels": hotels, "users": users}.items():
    print(f"\\n{name}: {frame.shape}")
    display(frame.dtypes.rename("dtype").to_frame())
    print("Missing values:", int(frame.isna().sum().sum()))
    print("Duplicate rows:", int(frame.duplicated().sum()))

display(flights.describe(include="all").T)

# 2. Classification — Predicting the Recorded Gender Label

## Objective and reasoning
The target is `users.gender`. For a reproducible baseline, this notebook uses `age` and `company`. It deliberately excludes `name` and `code`: names can act as direct identity proxies, while identifiers do not provide a defensible generalizable signal.

Gender is a sensitive attribute. This model is included to meet the academic specification only; it should not be used to infer a person's identity or to make consequential decisions. Metrics must be interpreted with attention to class imbalance and dataset bias.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

df = users.dropna(subset=["gender"]).copy()
df["gender"] = df["gender"].astype(str)
X = df[["age", "company"]].copy()
X["age"] = pd.to_numeric(X["age"], errors="coerce")
X["company"] = X["company"].fillna("unknown").astype(str)
y = df["gender"]
stratify = y if y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=stratify
)
prep = ColumnTransformer([
    ("company", OneHotEncoder(handle_unknown="ignore"), ["company"]),
    ("age", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), ["age"])
])
clf = Pipeline([("preprocess", prep),
                ("model", LogisticRegression(max_iter=1500, class_weight="balanced"))])
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0)
}
display(pd.DataFrame([metrics]))
print(classification_report(y_test, pred, zero_division=0))

## Evaluation and responsible interpretation

Accuracy alone can hide weak performance on a minority class. Balanced accuracy averages recall across classes, and macro F1 gives each class equal weight. Review the classification report and confusion matrix; compare results with a majority-class baseline before drawing conclusions. The available columns may not support meaningful prediction, and apparent predictive signal can reflect sampling or organizational bias rather than a reliable relationship.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45, cmap=None)
plt.title("Recorded-label classification — held-out test")
plt.tight_layout()
plt.show()

In [ ]:
import joblib
from pathlib import Path
Path("artifacts").mkdir(exist_ok=True)
joblib.dump(clf, "artifacts/gender_classifier.joblib")
print("Saved classifier to artifacts/gender_classifier.joblib")

## Deployment handoff
The repository's Flask API exposes a demonstration endpoint using only age and company and includes a warning in its response. Do not expose names, user identifiers, or other unnecessary personal data in API requests or logs. Add authentication and access controls before any shared deployment.